In [ ]:
!pip install --upgrade "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import spacy

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "state-spaces/mamba-130m-hf"


tokenizer = AutoTokenizer.from_pretrained(model_id)
nlp = spacy.load("en_core_web_sm")


class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):
        super().__init__()
        self.v_layer = nn.Linear(in_features, rank, bias=False)
        self.u_layer = nn.Linear(rank, out_features, bias=bias)
    def forward(self, x):
        return self.u_layer(self.v_layer(x))

def apply_svd_to_linear(module, rank):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name in ["in_proj", "out_proj"]:
            in_features = child.in_features
            out_features = child.out_features
            current_rank = min(rank, in_features, out_features)

            W = child.weight.data.float()
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            U_r, S_r, Vh_r = U[:, :current_rank], S[:current_rank], Vh[:current_rank, :]

            svd_module = SVDLinear(in_features, out_features, current_rank, bias=(child.bias is not None))
            svd_module.v_layer.weight.data = (torch.diag(S_r) @ Vh_r).to(child.weight.dtype)
            svd_module.u_layer.weight.data = U_r.to(child.weight.dtype)
            if child.bias is not None:
                svd_module.u_layer.bias.data = child.bias.data.clone()
            setattr(module, name, svd_module)
        else:
            apply_svd_to_linear(child, rank)


def load_my_model(peft_path, rank):
    print(f"\nLoading standard model...")
    base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map=device)

    print(f"Applying SVD (Rank {rank})...")
    apply_svd_to_linear(base_model, rank=rank)

    print(f"Loading LoRa weights: {peft_path}")
    model = PeftModel.from_pretrained(base_model, peft_path)
    model.eval()
    return model


def ask_model(model, prompt, use_tags=False):
    if use_tags:
        doc = nlp(prompt)
        final_prompt = " ".join([f"[{token.pos_}] {token.text}" for token in doc if token.text.strip()])
    else:
        final_prompt = prompt

    print(f"\n[PROMPT]: {final_prompt}")

    inputs = tokenizer(final_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            repetition_penalty=1.2
        )
    print(f"[OUTPUT]: {tokenizer.decode(output[0], skip_special_tokens=True)}")

In [ ]:
path_normal = "/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-64"
model_normal = load_my_model(path_normal, rank=64)



In [ ]:
path_pos = "/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-64-POS"
model_pos = load_my_model(path_pos, rank=64)



In [ ]:

wiki_prompt = "The Apollo 11 mission was the first manned flight to land on the Moon. The astronauts"

print("="*60)
print("Rank 64 no taggs")
print("="*60)
ask_model(model_normal, wiki_prompt, use_tags=False)

print("\n\n" + "="*60)
print("Rank 64 with taggs")
print("="*60)
ask_model(model_pos, wiki_prompt, use_tags=True)

In [ ]:
path_normal2 = "/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-256"
model_normal2 = load_my_model(path_normal2, rank=256)

In [ ]:
path_pos2 = "/content/drive/MyDrive/Project_Generative_Weights/mamba-wiki-text-final-rank-256-POS"
model_pos2 = load_my_model(path_pos2, rank=256)


In [ ]:
wiki_prompt = "The Apollo 11 mission was the first manned flight to land on the Moon. The astronauts"

print("="*60)
print("Rank 256 no taggs")
print("="*60)
ask_model(model_normal2, wiki_prompt, use_tags=False)

print("\n\n" + "="*60)
print("Rank 256 with taggs")
print("="*60)
ask_model(model_pos2, wiki_prompt, use_tags=True)

In [ ]:
wiki_prompt = "Elon Musk bought Tesla in the early 2000's and "
print("\n\n" + "="*60)
print("Rank 256 no taggs")
print("="*60)
ask_model(model_normal2, wiki_prompt, use_tags=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

ranks = ['Rank 526', 'Rank 256', 'Rank 64']
ppl_normal = [34.62, 85.06, 348.59]
ppl_pos = [0, 3.58, 4.44]

acc_normal = [41.0, 35.0, 18.1]
acc_pos = [0, 73.0, 71.13]

x = np.arange(len(ranks))
width = 0.35


fig, ax1 = plt.subplots(figsize=(8, 6))

rects1 = ax1.bar(x - width/2, ppl_normal, width, label='Normal (No Tags)', color='#e74c3c')

rects2 = ax1.bar(x[1:] + width/2, ppl_pos[1:], width, label='With POS Tags', color='#2ecc71')

ax1.set_ylabel('Perplexity (log scale)', fontweight='bold')
ax1.set_title('Perplexity per SVD Rank', fontweight='bold', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(ranks, fontweight='bold')
ax1.legend()


ax1.set_yscale('log')


ax1.bar_label(rects1, padding=3, fmt='%.1f')
ax1.bar_label(rects2, padding=3, fmt='%.1f')

fig.tight_layout()
plt.savefig('perplexity_chart.png', dpi=300)
plt.show()

print("\n" + "="*50 + "\n")


fig, ax2 = plt.subplots(figsize=(8, 6))

rects3 = ax2.bar(x - width/2, acc_normal, width, label='Normal (No Tags)', color='#3498db')
rects4 = ax2.bar(x[1:] + width/2, acc_pos[1:], width, label='With POS Tags', color='#9b59b6')

ax2.set_ylabel('Token Accuracy (%)', fontweight='bold')
ax2.set_title('POS Tags influence on accuracy', fontweight='bold', fontsize=14)
ax2.set_xticks(x)
ax2.set_xticklabels(ranks, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.legend()


ax2.bar_label(rects3, padding=3, fmt='%.1f%%')
ax2.bar_label(rects4, padding=3, fmt='%.1f%%')

fig.tight_layout()
plt.savefig('accuracy_chart.png', dpi=300)
plt.show()

In [ ]:
wiki_prompt = "Michael Schumacher was  one of the best F1 drivers in history and he is known for his 5 consequent titles with"

print("="*60)
print("Rank 64 no taggs")
print("="*60)
ask_model(model_normal, wiki_prompt, use_tags=False)

print("\n\n" + "="*60)
print("Rank 64 with taggs")
print("="*60)
ask_model(model_pos, wiki_prompt, use_tags=True)

In [ ]:
wiki_prompt = "Michael Schumacher was  one of the best F1 drivers in history and he is known for his 5 consequent titles with"

print("="*60)
print("Rank 256 no taggs")
print("="*60)
ask_model(model_normal2, wiki_prompt, use_tags=False)

print("\n\n" + "="*60)
print("Rank 256 with taggs")
print("="*60)
ask_model(model_pos2, wiki_prompt, use_tags=True)